# 37 · Query-term weights and field matching

Units: S04. Prerequisites: 36.

Sources: [Wang Shusen’s pinned outline](https://github.com/wangshusen/SearchEngine/tree/7339c6b126e314abdb74a89adb033d39d8e3d37b); [Manning et al., Introduction to Information Retrieval](https://nlp.stanford.edu/IR-book/) (supplementary original textbook, chapters 1, 2, 6, and 8). This is a Lab-authored exercise. Wang's last four modules have a public outline only; this is not presented as the author's complete lecture material.

Evidence boundary: a constructed English inventory and interpretable field scoring, not a complete BM25F implementation.

Learning method: predict each comparison, change one variable, run, and explain the difference.

> **English companion.** All instructional Markdown is translated. Experiment code, Chinese language examples, execution counts, and saved outputs remain as in the source edition; historical output text may therefore be Chinese. The initial setup cell anchors the working directory and Python import path for this companion's deeper folder; it does not change the experiment. Saved outputs are historical evidence, not a new run. Run from the top after preparing the required data and models.

In [ ]:
# English companions live one directory deeper; keep the original experiment code unchanged.
from pathlib import Path as _TranslationPath
import os as _translation_os
import sys as _translation_sys
_translation_root = next((p for p in [_TranslationPath.cwd(), *_TranslationPath.cwd().parents]
                          if (p / "reclab").is_dir()), None)
if _translation_root is None:
    raise RuntimeError("Open this notebook from inside the Search Recommendation Ads Lab checkout.")
_translation_os.chdir(_translation_root)
if str(_translation_root) not in _translation_sys.path:
    _translation_sys.path.insert(0, str(_translation_root))


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from reclab.learning.search_curriculum import *

## Inputs, outputs, and term weights
score=sum_q wq × (wt×title term frequency + body term frequency). Predict which documents rise when title weight increases. There is no length normalization; chapter 26 adds BM25 later.

In [2]:
q='apple repair'
for w in [0.,1.,4.]: print('title weight',w,field_rank(q,title_weight=w))

title weight 0.0 [('d', 2.0), ('b', 1.0), ('c', 1.0)]
title weight 1.0 [('d', 4.0), ('a', 2.0), ('b', 2.0), ('c', 2.0)]
title weight 4.0 [('d', 10.0), ('a', 8.0), ('b', 5.0), ('c', 5.0)]


## Repeating a query term does not make it mandatory
Fix documents and repeat only battery to increase its soft weight. Compare with the original BM25 contract, which binarizes query term frequency. This simple score is not BM25.

In [3]:
for q in ['apple repair battery','apple repair battery battery battery']:
    print(q,field_rank(q,title_weight=2))

apple repair battery [('d', 7), ('a', 5), ('c', 5), ('b', 4)]
apple repair battery battery battery [('c', 9), ('d', 9), ('a', 7), ('b', 6)]


## Hard constraints versus soft weights
Requiring official allows only documents containing that term. Predict what removes a high-scoring document that lacks it. A high score cannot compensate for a hard constraint.

In [4]:
soft=field_rank('apple repair'); hard=field_rank('apple repair',required=['official'])
print('soft',soft,'required official',hard)
assert [d for d,s in hard]==['a']

soft [('d', 6.0), ('a', 4.0), ('b', 3.0), ('c', 3.0)] required official [('a', 4.0)]


## Audit false retrievals
Artificial labels apply only to this task. Compute precision and inspect why fruit spam scores. Can changing only title weight fix every error?

In [5]:
relevant={'a','c','d'}
for w in [0.,4.]:
    order=[d for d,s in field_rank('battery repair',title_weight=w)][:2]
    print(w,order,'precision@2',len(set(order)&relevant)/2)

0.0 ['b', 'd'] precision@2 0.5
4.0 ['c', 'd'] precision@2 1.0


## Interpreting the saved run

For battery repair, increasing title weight from 0 to 4 changes top two from b/d to c/d and artificial precision@2 from 0.5 to 1, only on four constructed documents. Repeating battery promotes c; requiring official leaves only a. These illustrate distinct soft-weight and eligibility mechanisms.

After changing parameters, interpret the new outputs. This fixed commentary is not a new measurement.

## Review and exercise

Write down inputs, outputs, what stayed fixed, and conclusions this chapter cannot support. Predict and rerun one single-variable comparison above; retain failures. Return to the search learning path for the next semantic unit.